# Entrenamiento en Kaggle (modelos clásicos, sin redes neuronales)

Antes de correr:
1. **Add Input** → agrega el dataset combinado del equipo o, si aún no existe, `moltean/fruits` (Fruits-360).
2. **Settings** → Internet: **On** (requiere teléfono verificado). No hace falta GPU: todo corre en CPU.
3. Para correr sin tener la pestaña abierta: **Save Version → Save & Run All**.
4. **Add-ons → Secrets** → activa `GITHUB_TOKEN` para que el modelo se publique solo al final (ver docs/detalles.md, "Publicar el modelo"). Sin él, el notebook entrena igual pero no publica.

Este notebook solo clona el repo y llama a `src/`. Los cambios de código van en el repo, no aquí.

In [ ]:
import os, subprocess

REPO = "/kaggle/working/repo"
if not os.path.isdir(REPO):
    r = subprocess.run(["git", "clone", "https://github.com/11909/Clasificador-fruta-verdura.git", REPO],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("No se pudo clonar el repo. Activa Settings → Internet en el panel derecho "
                           "(Kaggle pide verificar tu teléfono) y vuelve a correr esta celda.\n" + r.stderr)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"])  # trae la última versión de main
%cd {REPO}
!pip install -q -r requirements.txt

Localiza las carpetas `Training` y `Test` del dataset adjunto y genera la config de esta corrida.

In [ ]:
import glob, yaml

import os

# Si está adjunto el dataset combinado del equipo (tiene manifest.csv) se usa ese; si no, Fruits-360.
equipo = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
if equipo:
    base = os.path.dirname(equipo[0])
    train_dirs, test_dirs = [f"{base}/train"], [f"{base}/test"]
else:
    train_dirs = sorted(glob.glob("/kaggle/input/**/Training", recursive=True))
    test_dirs = sorted(glob.glob("/kaggle/input/**/Test", recursive=True))
if not train_dirs or not test_dirs:
    print("Contenido de /kaggle/input:", glob.glob("/kaggle/input/*/*")[:20])
    raise FileNotFoundError("No encontré las carpetas de imágenes. Adjunta el dataset con Add Input "
                            "(Fruits-360: moltean/fruits, o el dataset combinado del equipo) y vuelve a correr.")
print("train:", train_dirs)
print("test:", test_dirs)

cfg = yaml.safe_load(open("configs/baseline.yaml"))
cfg["data"]["train_dirs"] = train_dirs[:1]   # ajusta si adjuntas más de un dataset
cfg["data"]["test_dirs"] = test_dirs[:1]
cfg["data"]["max_por_clase"] = 150           # súbelo para usar más imágenes (más lento)
cfg["data"]["balancear_etiquetas"] = True    # mismo número de frutas que de verduras
cfg["data"]["excluir_grupos"] = []           # ["nueces_semillas"] para sacar nueces y semillas
yaml.safe_dump(cfg, open("configs/run.yaml", "w"), allow_unicode=True)

Revisa que todas las clases tengan etiqueta en `data/mapping.csv`.

In [ ]:
import subprocess
r = subprocess.run(["python", "scripts/check_mapping.py", train_dirs[0]], capture_output=True, text=True)
print("\n".join(r.stdout.splitlines()[-5:]), r.stderr)


Extrae características, compara los modelos con validación cruzada, reentrena el mejor y lo evalúa en test.

In [ ]:
!python -m src.train --config configs/run.yaml

Resultados: comparación de modelos (`resultados_cv.csv`), métricas en test y el modelo guardado (`modelo.joblib`) en `/kaggle/working/outputs/`.

In [ ]:
import csv, json
for fila in csv.DictReader(open("/kaggle/working/outputs/resultados_cv.csv")):
    print(f"{fila['modelo']:20} f1_macro={fila['f1_macro']}  ±{fila['f1_macro_std']}  acc_balanceada={fila['balanced_accuracy']}  ({fila['segundos']} s)")
print(json.dumps(json.load(open("/kaggle/working/outputs/test_metrics.json")), indent=2))

## Revisar y publicar el modelo

Prueba rápida del modelo recién entrenado con las entradas de ejemplo. Después, la última celda lo publica sola como Release de GitHub, siempre que supere en accuracy balanceada de test al modelo publicado. Desde ese momento `notebooks/predecir.ipynb`, `python -m src.predict` y la interfaz usan el modelo nuevo.

In [ ]:
from src.clasificador import Clasificador

clf = Clasificador("/kaggle/working/outputs/modelo.joblib")
print(clf.descripcion)
for r in clf.clasificar_lote("entradas/ejemplos.txt"):
    print(r.get("etiqueta", "ERROR"), f"{r.get('confianza', 0):.0%}", r["entrada"].split("/")[-2:], r.get("error", ""))

In [ ]:
import os
from src.publicar import publicar

PUBLICAR = True   # False para entrenar sin publicar
FORZAR = False    # True publica aunque no supere al modelo publicado

token = os.environ.get("GITHUB_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None

if not PUBLICAR:
    print("Publicación desactivada (PUBLICAR = False).")
elif not token:
    print("No encontré el secreto GITHUB_TOKEN, así que el modelo no se publicó. "
          "Agrégalo en Add-ons → Secrets (ver docs/detalles.md, \"Publicar el modelo\") y corre esta celda otra vez.")
else:
    publicar("/kaggle/working/outputs", token, forzar=FORZAR)